# One-D-Piece Tokenizer Analysis
## Quality vs Token Count — SSIM, LPIPS, Composite QoE

**Purpose:** Characterise the One-D-Piece-L-256 tokenizer's rate-distortion behaviour.
Specifically, quantify how SSIM, LPIPS, and composite QoE vary as a function of
the number of transmitted tokens k — the core property that enables TCLA.

**Questions answered:**
1. Is QoE(k) truly monotone? Over what range? With what concavity?
2. What is the marginal quality gain per additional token at each operating point?
3. How does reconstruction quality compare to JPEG / WebP at equivalent bit rates?
4. Do SSIM and LPIPS agree? Where do they diverge?
5. What is the minimum viable k for a recognisable reconstruction?
6. How consistent is the QoE(k) curve across different image content?

**Runtime:** ~30 min on T4 GPU (LPIPS adds ~50ms per reconstruction).

In [ ]:
# CELL 1: Install packages
import subprocess, sys
print('Installing...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'einops', 'timm', 'omegaconf', 'scikit-image', 'lpips',
                'datasets', 'opencv-python-headless'],
               capture_output=True)
import torch, torchvision
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')
print('OK')

In [ ]:
# CELL 2: Clone repo + download weights
import subprocess, os, sys

REPO_DIR = '/content/One-D-Piece'
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(['git','clone','--depth','1',
                        'https://github.com/turingmotors/One-D-Piece.git', REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout or 'cloned')
else:
    print('Repo already present.')
sys.path.insert(0, REPO_DIR)

WEIGHT = '/content/pytorch_model.bin'
HF_URL = 'https://huggingface.co/turing-motors/One-D-Piece-L-256/resolve/main/pytorch_model.bin'
if os.path.isfile(WEIGHT) and os.path.getsize(WEIGHT) > 1e9:
    print(f'Weights present ({os.path.getsize(WEIGHT)/1e9:.2f} GB)')
else:
    print('Downloading weights (~2.6 GB)...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', WEIGHT, HF_URL])
    print(f'Done: {os.path.getsize(WEIGHT)/1e9:.2f} GB')

In [ ]:
# CELL 3: Load model
import sys, torch, numpy as np
sys.path.insert(0, '/content/One-D-Piece')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

from omegaconf import OmegaConf
from modeling.titok import TiTok

cfg = OmegaConf.create({'model': {'vq_model': {
    'codebook_size': 4096, 'token_size': 12,
    'use_l2_norm': True, 'commitment_cost': 0.25,
    'num_latent_tokens': 256,
    'vit_enc_model_size': 'large', 'vit_dec_model_size': 'large',
    'vit_enc_patch_size': 16, 'vit_dec_patch_size': 16,
    'finetune_decoder': True,
}}, 'dataset': {'preprocessing': {'crop_size': 256}}})

model = TiTok(cfg)
ckpt  = torch.load('/content/pytorch_model.bin', map_location='cpu', weights_only=False)
state = ckpt.get('state_dict', ckpt.get('model', ckpt))
keys  = list(state.keys())
for prefix in ['model.', 'module.', 'vq_model.']:
    if sum(1 for k in keys[:10] if k.startswith(prefix)) >= 5:
        state = {k[len(prefix):]: v for k,v in state.items()}
        print(f'Stripped prefix: "{prefix}"'); break
m, u = model.load_state_dict(state, strict=False)
print(f'Missing: {len(m)}  Unexpected: {len(u)}')
model = model.to(DEVICE).eval()
print(f'Ready — {sum(p.numel() for p in model.parameters())/1e6:.0f}M params')

In [ ]:
# CELL 4: Core functions — tokenize, reconstruct, measure
import torch, numpy as np
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
import lpips as lpips_lib

_lpips_fn = None
def get_lpips():
    global _lpips_fn
    if _lpips_fn is None:
        _lpips_fn = lpips_lib.LPIPS(net='alex').to(DEVICE); _lpips_fn.eval()
    return _lpips_fn

TO_TENSOR = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.LANCZOS),
    T.CenterCrop(256), T.ToTensor(),
])

def pil_to_tensor(pil):
    if pil.mode != 'RGB': pil = pil.convert('RGB')
    return TO_TENSOR(pil).unsqueeze(0).to(DEVICE)

def tensor_to_pil(t):
    arr = t.squeeze(0).permute(1,2,0).cpu().float().numpy()
    return Image.fromarray((np.clip(arr,0,1)*255).astype(np.uint8))

@torch.no_grad()
def tokenize(pil_img):
    _, info = model.encode(pil_to_tensor(pil_img))
    return info['min_encoding_indices'].squeeze().cpu().numpy().astype(np.int64)

@torch.no_grad()
def reconstruct(token_ids, k):
    k = max(1, min(int(k), 256))
    padded = np.zeros(256, dtype=np.int64); padded[:k] = token_ids[:k]
    return tensor_to_pil(model.decode_tokens(torch.tensor(padded).reshape(1,1,256).to(DEVICE)))

def measure_all(orig_pil, rec_pil):
    """Compute PSNR, SSIM, LPIPS, and composite QoE."""
    o = np.array(orig_pil.convert('RGB').resize((256,256), Image.LANCZOS))
    r = np.array(rec_pil.convert('RGB').resize((256,256), Image.LANCZOS))
    psnr_val = float(psnr_fn(o, r, data_range=255))
    ssim_val = float(ssim_fn(o, r, channel_axis=2, data_range=255))
    o_t = TF.to_tensor(Image.fromarray(o)).unsqueeze(0).to(DEVICE)*2-1
    r_t = TF.to_tensor(Image.fromarray(r)).unsqueeze(0).to(DEVICE)*2-1
    with torch.no_grad():
        lpips_val = float(get_lpips()(o_t, r_t).item())
    lpips_qoe = max(0.0, 1.0 - lpips_val)
    composite  = 0.5*ssim_val + 0.5*lpips_qoe
    return dict(psnr=psnr_val, ssim=ssim_val, lpips=lpips_val,
                lpips_qoe=lpips_qoe, composite=composite)

# Smoke test
from skimage import data as skdata
tpil = Image.fromarray(skdata.astronaut())
toks = tokenize(tpil)
m    = measure_all(tpil, reconstruct(toks, 64))
print(f'Smoke: k=64  PSNR={m["psnr"]:.1f}dB  SSIM={m["ssim"]:.3f}  '
      f'LPIPS={m["lpips"]:.3f}  Composite={m["composite"]:.3f}')
print('OK')

In [ ]:
# CELL 5: Load test images
# Uses both Tiny-ImageNet (in-distribution) and skimage (OOD) for comparison
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def arr2pil(a):
    if a.ndim==2: a=np.stack([a,a,a],axis=-1)
    if a.dtype!=np.uint8: a=(a*255).clip(0,255).astype(np.uint8)
    return Image.fromarray(a).convert('RGB')

ind_images = []   # in-distribution (Tiny-ImageNet)
ood_images = []   # out-of-distribution (skimage)

try:
    from datasets import load_dataset
    print('Loading Tiny-ImageNet (in-distribution)...')
    ds = load_dataset('zh-plus/tiny-imagenet', split='valid',
                      streaming=True, trust_remote_code=True)
    for i, s in enumerate(ds):
        if len(ind_images) >= 10: break
        img = s['image'].convert('RGB').resize((256,256), Image.LANCZOS)
        ind_images.append((f'IND_{i}', img))
    print(f'Loaded {len(ind_images)} IND images')
except Exception as e:
    print(f'Tiny-ImageNet failed ({e})')

from skimage import data as skdata
ood_images = [
    ('astronaut', arr2pil(skdata.astronaut())),
    ('chelsea',   arr2pil(skdata.chelsea())),
    ('coffee',    arr2pil(skdata.coffee())),
    ('camera',    arr2pil(skdata.camera())),
    ('coins',     arr2pil(skdata.coins())),
    ('horse',     arr2pil(skdata.horse())),
    ('rocket',    arr2pil(skdata.rocket())),
    ('page',      arr2pil(skdata.page())),
]
print(f'Loaded {len(ood_images)} OOD images (skimage)')

all_images = ind_images + ood_images
print(f'\nTotal: {len(all_images)} images ({len(ind_images)} IND + {len(ood_images)} OOD)')

# Display sample
show_n = min(8, len(all_images))
fig, axes = plt.subplots(1, show_n, figsize=(3*show_n, 3), facecolor='#0d0f1a')
for ax, (name, img) in zip(axes, all_images[:show_n]):
    ax.imshow(img.resize((128,128), Image.LANCZOS))
    ax.set_title(name[:12], color='white', fontsize=7); ax.axis('off')
plt.tight_layout(); plt.show()
print('Images ready.')

In [ ]:
# CELL 6: Dense QoE(k) measurement for all images
# Evaluates SSIM, LPIPS, PSNR, Composite at every k in K_EVAL
# ~30 min on T4 depending on number of images
import numpy as np

# Dense k values — finer at low k where curve changes fastest
K_EVAL = sorted(set(
    list(range(1, 17, 1))      +   # k=1..16 every 1
    list(range(16, 65, 4))     +   # k=16..64 every 4
    list(range(64, 257, 8))        # k=64..256 every 8
))
print(f'Evaluating {len(K_EVAL)} k-values per image')
print(f'k range: {K_EVAL[0]}..{K_EVAL[-1]}')

results = {}   # results[img_name] = dict of lists

for img_name, pil_img in all_images:
    print(f'\n{img_name}  (tokenizing...)', end=' ', flush=True)
    toks = tokenize(pil_img)
    print(f'range=[{toks.min()},{toks.max()}]  measuring {len(K_EVAL)} points...')

    r = dict(k=[], psnr=[], ssim=[], lpips=[], lpips_qoe=[], composite=[])
    for ki, k in enumerate(K_EVAL):
        m = measure_all(pil_img, reconstruct(toks, k))
        r['k'].append(k)
        for key in ['psnr','ssim','lpips','lpips_qoe','composite']:
            r[key].append(m[key])
        if ki % 20 == 0:
            print(f'  k={k:3d}: PSNR={m["psnr"]:5.1f}dB  SSIM={m["ssim"]:.3f}  '
                  f'LPIPS={m["lpips"]:.3f}  Comp={m["composite"]:.3f}')

    for key in r: r[key] = np.array(r[key])
    r['toks'] = toks
    r['pil']  = pil_img
    r['is_ind'] = img_name.startswith('IND_')

    # Check monotonicity
    comp = r['composite']
    viol = sum(1 for i in range(len(comp)-1) if comp[i] > comp[i+1]+0.01)
    print(f'  Monotone violations (>0.01): {viol}/{len(K_EVAL)-1}  '
          f'k=1:{comp[0]:.3f} k=32:{comp[list(K_EVAL).index(32)]:.3f} '
          f'k=128:{comp[list(K_EVAL).index(128)]:.3f} '
          f'k=256:{comp[-1]:.3f}')

    results[img_name] = r

print(f'\nAll done. {len(results)} images processed.')

In [ ]:
# CELL 7: Compute JPEG / WebP baselines at equivalent bit rates
# Token budget k → bits = k × 12  →  find JPEG quality giving same file size
import numpy as np, io, cv2
from PIL import Image

BITS_PER_TOKEN = 12
K_BASELINE     = [8, 16, 32, 64, 128, 192, 256]

def encode_jpeg(pil_img, quality):
    buf = io.BytesIO()
    pil_img.convert('RGB').resize((256,256),Image.LANCZOS).save(buf, 'JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).copy(), buf.tell()

def encode_webp(pil_img, quality):
    buf = io.BytesIO()
    pil_img.convert('RGB').resize((256,256),Image.LANCZOS).save(buf, 'WebP', quality=quality)
    buf.seek(0)
    return Image.open(buf).copy(), buf.tell()

def find_quality_for_bits(pil_img, target_bits, fmt='JPEG', q_range=(1,95)):
    """Binary search for codec quality level matching target bit budget."""
    lo, hi = q_range
    for _ in range(10):
        mid = (lo + hi) // 2
        fn  = encode_jpeg if fmt=='JPEG' else encode_webp
        _, size = fn(pil_img, mid)
        if size*8 < target_bits: lo = mid
        else:                    hi = mid
    fn = encode_jpeg if fmt=='JPEG' else encode_webp
    return fn(pil_img, (lo+hi)//2)

# Compute baselines on each image
baselines = {}
for img_name, pil_img in list(all_images)[:min(5, len(all_images))]:
    print(f'{img_name}:')
    bsl = dict(k=K_BASELINE, jpeg_comp=[], webp_comp=[], jpeg_bpp=[], webp_bpp=[])
    for k in K_BASELINE:
        target_bits = k * BITS_PER_TOKEN
        j_img, j_bytes = find_quality_for_bits(pil_img, target_bits, 'JPEG')
        w_img, w_bytes = find_quality_for_bits(pil_img, target_bits, 'WebP')
        mj = measure_all(pil_img, j_img)
        mw = measure_all(pil_img, w_img)
        bsl['jpeg_comp'].append(mj['composite'])
        bsl['webp_comp'].append(mw['composite'])
        bsl['jpeg_bpp'].append(j_bytes*8/256/256)
        bsl['webp_bpp'].append(w_bytes*8/256/256)
        print(f'  k={k:3d} ({target_bits:5d} bits):  '
              f'JPEG={mj["composite"]:.3f}  WebP={mw["composite"]:.3f}  '
              f'ODP={results[img_name]["composite"][list(K_EVAL).index(k)]:.3f}')
    baselines[img_name] = bsl

print('\nBaseline comparison done.')

In [ ]:
# CELL 8: Marginal quality gain analysis
# dQoE/dk — how much quality does each additional token contribute?
# This reveals the saturation point and the "knee" of the curve.
import numpy as np

print('Marginal quality analysis (dComposite/dk):')
print(f'{"Image":<18} {"Knee_k":>8} {"Max_gain/tok":>14} {"Gain_1to32":>12} '
      f'{"Gain_32to128":>14} {"Gain_128to256":>15}')
print('-'*85)

marginals = {}
for img_name, r in results.items():
    k    = r['k']
    comp = r['composite']

    # Compute marginal gain per token (finite difference)
    dk   = np.diff(k)
    dq   = np.diff(comp)
    dqdk = dq / dk   # quality gain per token

    # Find the "knee" — k where marginal gain drops below 10% of peak
    peak_rate = dqdk.max()
    knee_idx  = np.argmax(dqdk < 0.1 * peak_rate)
    knee_k    = int(k[knee_idx+1]) if knee_idx < len(k)-1 else int(k[-1])

    # Total gain in different ranges
    def gain_range(ka, kb):
        ia = np.searchsorted(k, ka); ib = np.searchsorted(k, kb)
        return float(comp[ib] - comp[ia]) if ib < len(comp) else float(comp[-1]-comp[ia])

    g1_32   = gain_range(1, 32)
    g32_128 = gain_range(32, 128)
    g128_256= gain_range(128, 256)

    marginals[img_name] = dict(dqdk=dqdk, k_mid=k[1:], knee_k=knee_k,
                                g1_32=g1_32, g32_128=g32_128, g128_256=g128_256)
    print(f'{img_name:<18} {knee_k:>8d} {peak_rate*100:>13.2f}% '
          f'{g1_32:>12.3f} {g32_128:>14.3f} {g128_256:>15.3f}')

print('\nNote: Gain shown as absolute composite QoE increase in each token range.')
print('Knee_k: first k where marginal gain drops below 10% of peak rate.')

In [ ]:
# CELL 9: Token importance analysis
# For each token position i, measure the quality DROP when only that token is zeroed.
# Confirms TTD ordering: early tokens should have largest impact.
# (Runs on primary image only — takes ~5 min)
import numpy as np

primary_name = list(results.keys())[0]
r_prim       = results[primary_name]
toks_prim    = r_prim['toks']
pil_prim     = r_prim['pil']

# Baseline: all 256 tokens
full_rec     = reconstruct(toks_prim, 256)
full_m       = measure_all(pil_prim, full_rec)
print(f'Full reconstruction ({primary_name}): Composite={full_m["composite"]:.3f}')

# Ablate each token position (set to 0) and measure QoE drop
ABLATE_POSITIONS = list(range(0, 256, 4))   # every 4th position for speed
print(f'Ablating {len(ABLATE_POSITIONS)} token positions...')

ablation = dict(pos=[], comp_drop=[], ssim_drop=[], lpips_increase=[])
for pos in ABLATE_POSITIONS:
    ablated = toks_prim.copy(); ablated[pos] = 0
    m_abl   = measure_all(pil_prim, reconstruct(ablated, 256))
    ablation['pos'].append(pos)
    ablation['comp_drop'].append(full_m['composite']  - m_abl['composite'])
    ablation['ssim_drop'].append(full_m['ssim']       - m_abl['ssim'])
    ablation['lpips_increase'].append(m_abl['lpips']  - full_m['lpips'])
    if pos % 32 == 0:
        print(f'  pos={pos:3d}: comp_drop={ablation["comp_drop"][-1]:+.4f}  '
              f'ssim_drop={ablation["ssim_drop"][-1]:+.4f}')

for k in ablation: ablation[k] = np.array(ablation[k])
print(f'\nTTD check: mean|drop| positions 0-31:   {ablation["comp_drop"][:8].mean():.4f}')
print(f'TTD check: mean|drop| positions 128-191: {ablation["comp_drop"][32:48].mean():.4f}')
print(f'TTD check: mean|drop| positions 224-255: {ablation["comp_drop"][56:].mean():.4f}')
print('(Head tokens should have larger drop than tail — confirms TTD ordering)')

In [ ]:
# CELL 10: SSIM vs LPIPS agreement analysis
# Identifies the k range where the two metrics diverge most.
# Divergence indicates that one metric is capturing information the other misses.
import numpy as np

print('SSIM vs LPIPS agreement (Pearson r per image):')
print(f'{"Image":<18} {"r(SSIM,LPIPS_QoE)":>20} {"Max_divergence":>16} {"Diverge_at_k":>14}')
print('-'*72)

agreement = {}
for img_name, r in results.items():
    ssim = r['ssim']; lqoe = r['lpips_qoe']; k = r['k']

    # Normalise both to [0,1]
    sn = (ssim - ssim.min()) / (ssim.max()-ssim.min()+1e-8)
    ln = (lqoe - lqoe.min()) / (lqoe.max()-lqoe.min()+1e-8)

    corr     = float(np.corrcoef(sn, ln)[0,1])
    diff     = np.abs(sn - ln)
    max_div  = float(diff.max())
    div_k    = int(k[diff.argmax()])

    agreement[img_name] = dict(corr=corr, sn=sn, ln=ln, diff=diff)
    print(f'{img_name:<18} {corr:>20.4f} {max_div:>16.4f} {div_k:>14d}')

print('\nInterpretation:')
print('  r close to 1.0 → SSIM and LPIPS agree — robust quality signal')
print('  Large divergence at low k → SSIM sees structure, LPIPS sees perceptual blur')
print('  Large divergence at high k → LPIPS sensitive to texture; SSIM is not')

In [ ]:
# CELL 11: IND vs OOD quality comparison
# One-D-Piece was trained on ImageNet. Tiny-ImageNet is in-distribution (IND).
# Skimage images are out-of-distribution (OOD). Quantify the quality gap.
import numpy as np

K_CHECK = [16, 32, 64, 128, 256]

ind_names = [n for n,r in results.items() if r['is_ind']]
ood_names = [n for n,r in results.items() if not r['is_ind']]

if ind_names:
    print('IND (Tiny-ImageNet) vs OOD (skimage) quality comparison:')
    print(f'{"k":>6}  {"IND_SSIM":>10} {"OOD_SSIM":>10} {"Gap":>8}  '
          f'{"IND_LPIPS":>11} {"OOD_LPIPS":>11} {"Gap":>8}  '
          f'{"IND_Comp":>10} {"OOD_Comp":>10} {"Gap":>8}')
    print('-'*100)

    for k in K_CHECK:
        ki = list(K_EVAL).index(k) if k in K_EVAL else None
        if ki is None: continue
        ind_ssim  = np.mean([results[n]['ssim'][ki]  for n in ind_names])
        ood_ssim  = np.mean([results[n]['ssim'][ki]  for n in ood_names])
        ind_lpips = np.mean([results[n]['lpips'][ki] for n in ind_names])
        ood_lpips = np.mean([results[n]['lpips'][ki] for n in ood_names])
        ind_comp  = np.mean([results[n]['composite'][ki] for n in ind_names])
        ood_comp  = np.mean([results[n]['composite'][ki] for n in ood_names])
        print(f'{k:>6}  {ind_ssim:>10.3f} {ood_ssim:>10.3f} '
              f'{ind_ssim-ood_ssim:>+8.3f}  '
              f'{ind_lpips:>11.3f} {ood_lpips:>11.3f} '
              f'{ind_lpips-ood_lpips:>+8.3f}  '
              f'{ind_comp:>10.3f} {ood_comp:>10.3f} '
              f'{ind_comp-ood_comp:>+8.3f}')
else:
    print('No IND images loaded (Tiny-ImageNet unavailable). Skipping IND/OOD comparison.')
    print('Only OOD (skimage) results available.')

In [ ]:
# CELL 12: Visual reconstruction strip — all k values for primary image
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

BG='#0d0f1a'; WH='#e8e8f0'; GR='#aaaacc'

primary_name = list(results.keys())[0]
r_prim = results[primary_name]
toks_p = r_prim['toks']
pil_p  = r_prim['pil']

k_show = [1, 4, 8, 16, 24, 32, 48, 64, 96, 128, 160, 192, 224, 256]
nk     = len(k_show)

fig, axes = plt.subplots(3, nk, figsize=(nk*2.2, 7.5), facecolor=BG)
fig.suptitle(f'One-D-Piece Reconstruction vs Token Count — {primary_name}',
             color=WH, fontsize=12, fontweight='bold', y=0.98)

k_arr = np.array(r_prim['k'])
for col, k in enumerate(k_show):
    rec   = reconstruct(toks_p, k)
    ki    = np.argmin(np.abs(k_arr - k))
    ssim  = r_prim['ssim'][ki]
    lpips = r_prim['lpips'][ki]
    comp  = r_prim['composite'][ki]

    # Row 0: reconstructed image
    axes[0,col].imshow(np.array(rec.resize((128,128), Image.LANCZOS)))
    axes[0,col].axis('off')
    axes[0,col].set_title(f'k={k}', color=WH, fontsize=8, pad=2)

    # Row 1: difference map vs original
    orig_np = np.array(pil_p.resize((256,256), Image.LANCZOS))
    rec_np  = np.array(rec.resize((256,256), Image.LANCZOS))
    diff    = np.abs(orig_np.astype(float) - rec_np.astype(float)).mean(axis=2)
    axes[1,col].imshow(diff, cmap='hot', vmin=0, vmax=60)
    axes[1,col].axis('off')
    axes[1,col].set_title(f'|diff|\nmean={diff.mean():.1f}', color=GR, fontsize=6.5, pad=2)

    # Row 2: metric values
    axes[2,col].set_facecolor('#13162a')
    axes[2,col].text(0.5, 0.75, f'SSIM\n{ssim:.3f}',  ha='center', va='center',
                     color='#4FC3F7', fontsize=7, transform=axes[2,col].transAxes)
    axes[2,col].text(0.5, 0.40, f'LPIPS\n{lpips:.3f}', ha='center', va='center',
                     color='#EF5350', fontsize=7, transform=axes[2,col].transAxes)
    axes[2,col].text(0.5, 0.08, f'QoE\n{comp:.3f}',   ha='center', va='center',
                     color='#66BB6A', fontsize=7, transform=axes[2,col].transAxes,
                     fontweight='bold')
    axes[2,col].axis('off')

plt.tight_layout(rect=[0,0,1,0.97])
plt.savefig('/content/odp_reconstruction_strip.png', dpi=130,
            bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: odp_reconstruction_strip.png')

In [ ]:
# CELL 13: Main analysis figure
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

BG='#0d0f1a'; PAN='#13162a'; GR='#aaaacc'; WH='#e8e8f0'
TC='#4FC3F7'; CV='#EF5350'; GN='#66BB6A'; OR='#FFA726'; PR='#CE93D8'; YL='#FFD54F'
IND_COL = GN; OOD_COL = TC

def stl(ax, t=''):
    ax.set_facecolor(PAN); ax.tick_params(colors=GR, labelsize=8)
    for s in ['bottom','left']: ax.spines[s].set_color('#334')
    for s in ['top','right']:   ax.spines[s].set_visible(False)
    ax.grid(True, color='#1e2235', lw=0.6)
    ax.xaxis.label.set_color(GR); ax.yaxis.label.set_color(GR)
    if t: ax.set_title(t, color=WH, fontsize=8.5, fontweight='bold', pad=5)

fig = plt.figure(figsize=(22, 20), facecolor=BG)
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.50, wspace=0.35,
                        left=0.06, right=0.975, top=0.935, bottom=0.04)
fig.text(0.5, 0.958, 'One-D-Piece-L-256 Tokenizer: Quality vs Token Count Analysis',
         ha='center', color=WH, fontsize=14, fontweight='bold')
fig.text(0.5, 0.937,
         'QoE = 0.5·SSIM + 0.5·(1−LPIPS)  |  256×256 images  |  '
         'Codebook: 4096 entries × 12 bits = 3,072 bits at k=256',
         ha='center', color=GR, fontsize=8)

cm = plt.colormaps['tab20']
k_arr = np.array(list(results.values())[0]['k'])

# ── Row 1: SSIM / LPIPS / Composite curves ───────────────────────────
for col, (metric, ylabel, title, inv) in enumerate([
    ('ssim',      'SSIM',       'SSIM vs Token Count',      False),
    ('lpips',     'LPIPS',      'LPIPS vs Token Count\n(lower = better)', True),
    ('composite', 'Composite QoE (0.5·SSIM + 0.5·(1−LPIPS))',
     'Composite QoE vs Token Count', False),
]):
    ax = fig.add_subplot(gs[0, col]); stl(ax, title)
    for i,(name,r) in enumerate(results.items()):
        col_c = cm(i % 20)
        ls    = '-' if r['is_ind'] else '--'
        lw    = 1.8 if r['is_ind'] else 1.2
        ax.plot(r['k'], r[metric], color=col_c, lw=lw, ls=ls, alpha=0.85,
                label=name if i < 6 else None)
    # Mean IND / OOD
    ind_ns = [n for n,r in results.items() if r['is_ind']]
    ood_ns = [n for n,r in results.items() if not r['is_ind']]
    if ind_ns:
        ax.plot(k_arr, np.mean([results[n][metric] for n in ind_ns],axis=0),
                color=IND_COL, lw=3, ls='-', alpha=0.9, label='Mean IND')
    if ood_ns:
        ax.plot(k_arr, np.mean([results[n][metric] for n in ood_ns],axis=0),
                color=OOD_COL, lw=3, ls='-', alpha=0.9, label='Mean OOD')
    ax.set_xlabel('Token count k'); ax.set_ylabel(ylabel)
    ax.set_xlim(0, 256)
    if not inv: ax.set_ylim(0, 1.0)
    ax.legend(fontsize=6.5, facecolor=BG, edgecolor='#334', labelcolor=WH,
              ncol=2, loc='lower right')

# ── Row 2L: Marginal gain per token (dComposite/dk) ──────────────────
ax = fig.add_subplot(gs[1, 0]); stl(ax, 'Marginal QoE Gain per Token (dComposite/dk)\nReveals saturation point and "knee" of curve')
for i, (name, m) in enumerate(marginals.items()):
    r = results[name]
    ls = '-' if r['is_ind'] else '--'
    ax.plot(m['k_mid'], np.maximum(m['dqdk'], 0)*100, color=cm(i%20),
            lw=1.4, ls=ls, alpha=0.8, label=name if i<4 else None)
ax.set_xlabel('Token count k'); ax.set_ylabel('Marginal gain per token (%)')
ax.set_xlim(0, 256); ax.set_ylim(bottom=0)
ax.legend(fontsize=6.5, facecolor=BG, edgecolor='#334', labelcolor=WH)
ax.axvline(32, color=OR, lw=1, ls=':', alpha=0.6, label='k=32 (base layer)')

# ── Row 2M: SSIM vs LPIPS scatter per image ──────────────────────────
ax = fig.add_subplot(gs[1, 1]); stl(ax, 'SSIM vs LPIPS Correlation per Image\nDivergence regions reveal where metrics disagree')
for i, (name, r) in enumerate(results.items()):
    sc = ax.scatter(r['ssim'], r['lpips_qoe'], c=r['k'], cmap='viridis',
                    s=12, alpha=0.6, marker='o' if r['is_ind'] else 's')
ax.plot([0,1],[0,1], color=WH, lw=0.8, ls='--', alpha=0.4, label='Perfect agreement')
ax.set_xlabel('SSIM'); ax.set_ylabel('1 − LPIPS')
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.colorbar(sc, ax=ax, label='k').ax.tick_params(colors=GR, labelsize=7)
ax.legend(fontsize=7, facecolor=BG, edgecolor='#334', labelcolor=WH)

# ── Row 2R: ODP vs JPEG vs WebP comparison ───────────────────────────
ax = fig.add_subplot(gs[1, 2]); stl(ax, 'One-D-Piece vs JPEG vs WebP\nAt equivalent bit budget (k × 12 bits)')
if baselines:
    bn = list(baselines.keys())[0]
    bl = baselines[bn]; r_b = results[bn]
    k_b = np.array(bl['k'])
    comp_odp = np.array([r_b['composite'][list(K_EVAL).index(k)] for k in bl['k']])
    ax.plot(k_b, comp_odp,            color=GN,  lw=2.2, marker='o', ms=5, label='One-D-Piece')
    ax.plot(k_b, bl['jpeg_comp'],     color=OR,  lw=1.8, marker='s', ms=5, label='JPEG')
    ax.plot(k_b, bl['webp_comp'],     color=TC,  lw=1.8, marker='^', ms=5, label='WebP')
    ax.set_xlabel('Token count k (=bits/12)'); ax.set_ylabel('Composite QoE')
    ax.set_xlim(0, max(bl['k'])+10); ax.set_ylim(0, 1.0)
    ax.legend(facecolor=BG, edgecolor='#334', labelcolor=WH, fontsize=9)
    ax.set_title(ax.get_title() + f'\n({bn})', color=WH, fontsize=8.5, fontweight='bold', pad=5)

# ── Row 3: Token ablation importance map ─────────────────────────────
ax = fig.add_subplot(gs[2, 0:2])
stl(ax, f'Token Importance by Ablation — {primary_name}\n'
        'QoE drop when each position is zeroed — confirms TTD ordering (early tokens most important)')
pos  = ablation['pos']
cdrop= ablation['comp_drop']
sdrop= ablation['ssim_drop']
linc = ablation['lpips_increase']
ax.bar(pos, cdrop*100,  width=3, color=GN,  alpha=0.85, label='Composite QoE drop (%)')
ax.plot(pos, sdrop*100, color=TC,  lw=1.5, alpha=0.8, label='SSIM drop (%)')
ax.plot(pos, linc*100,  color=CV,  lw=1.5, alpha=0.8, label='LPIPS increase (%)')
ax.axvline(32,  color=OR, lw=1, ls=':', alpha=0.6, label='k_base=32')
ax.axvline(128, color=PR, lw=1, ls=':', alpha=0.6, label='k=128')
ax.set_xlabel('Token position (ablated)'); ax.set_ylabel('Quality impact (%)')
ax.set_xlim(0, 256)
ax.legend(fontsize=7.5, facecolor=BG, edgecolor='#334', labelcolor=WH)

# ── Row 3R: SSIM vs LPIPS divergence ─────────────────────────────────
ax = fig.add_subplot(gs[2, 2])
stl(ax, 'SSIM vs (1−LPIPS) Normalised — Per Image\nShows where metrics diverge across k range')
for i, (name, ag) in enumerate(agreement.items()):
    ax.plot(k_arr, ag['sn'], color=cm(i%20), lw=1.5, ls='-', alpha=0.7)
    ax.plot(k_arr, ag['ln'], color=cm(i%20), lw=1.5, ls='--', alpha=0.7)
from matplotlib.lines import Line2D
handles = [Line2D([0],[0],color=WH,lw=1.5,ls='-',  label='Normalised SSIM'),
           Line2D([0],[0],color=WH,lw=1.5,ls='--', label='Normalised 1−LPIPS')]
ax.set_xlabel('Token count k'); ax.set_ylabel('Normalised metric [0,1]')
ax.set_xlim(0,256); ax.set_ylim(0,1.0)
ax.legend(handles=handles, facecolor=BG, edgecolor='#334', labelcolor=WH, fontsize=8)

# ── Row 4: Key operating points summary ──────────────────────────────
ax = fig.add_subplot(gs[3, 0:3])
stl(ax, 'Operating Points Summary — Mean Composite QoE at Key Token Counts\n'
        'with ±1σ band across all images (IND=solid, OOD=dashed)')
all_comp = np.array([r['composite'] for r in results.values()])
mean_comp= all_comp.mean(axis=0)
std_comp = all_comp.std(axis=0)
ax.fill_between(k_arr, mean_comp-std_comp, mean_comp+std_comp,
                alpha=0.15, color=WH, label='±1σ all images')
ax.plot(k_arr, mean_comp, color=WH, lw=2.5, label='Mean all images')
if ind_ns:
    ind_comp = np.array([results[n]['composite'] for n in ind_ns]).mean(axis=0)
    ax.plot(k_arr, ind_comp, color=IND_COL, lw=2, ls='-', label=f'Mean IND ({len(ind_ns)} images)')
if ood_ns:
    ood_comp = np.array([results[n]['composite'] for n in ood_ns]).mean(axis=0)
    ax.plot(k_arr, ood_comp, color=OOD_COL, lw=2, ls='--', label=f'Mean OOD ({len(ood_ns)} images)')
for k_mark, col, lbl in [(32,OR,'k_base=32'),(64,PR,'k=64'),(128,YL,'k=128'),(256,GN,'k=256')]:
    ki = np.argmin(np.abs(k_arr-k_mark))
    ax.axvline(k_mark, color=col, lw=1, ls=':', alpha=0.7)
    ax.text(k_mark, mean_comp[ki]+0.03, f'{mean_comp[ki]:.3f}', color=col,
            ha='center', fontsize=8)
ax.set_xlabel('Token count k'); ax.set_ylabel('Composite QoE')
ax.set_xlim(0,256); ax.set_ylim(0,1.0)
ax.legend(facecolor=BG, edgecolor='#334', labelcolor=WH, fontsize=8, ncol=2)

plt.savefig('/content/odp_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: odp_analysis.png')

In [ ]:
# CELL 14: Summary report + download all figures
import numpy as np

print('='*72)
print('ONE-D-PIECE-L-256 TOKENIZER ANALYSIS — Summary')
print('='*72)
print(f'Images analysed: {len(results)} '
      f'({len([n for n,r in results.items() if r["is_ind"]])} IND, '
      f'{len([n for n,r in results.items() if not r["is_ind"]])} OOD)')
print(f'k values evaluated: {len(K_EVAL)}  (k=1..256)')

# Aggregate metrics at key k values
K_REPORT = [1, 8, 16, 32, 48, 64, 96, 128, 192, 256]
k_arr = np.array(list(results.values())[0]['k'])
print(f'\nMean metrics across all images:')
print(f'{"k":>5}  {"PSNR":>8}  {"SSIM":>8}  {"LPIPS":>8}  '
      f'{"1-LPIPS":>9}  {"Composite":>11}  {"Std_Comp":>10}')
print('-'*72)
for k in K_REPORT:
    ki = np.argmin(np.abs(k_arr-k))
    p  = np.mean([r['psnr'][ki]      for r in results.values()])
    s  = np.mean([r['ssim'][ki]      for r in results.values()])
    l  = np.mean([r['lpips'][ki]     for r in results.values()])
    lq = np.mean([r['lpips_qoe'][ki] for r in results.values()])
    c  = np.mean([r['composite'][ki] for r in results.values()])
    cs = np.std( [r['composite'][ki] for r in results.values()])
    print(f'{k:>5}  {p:>8.2f}  {s:>8.3f}  {l:>8.3f}  {lq:>9.3f}  {c:>11.3f}  {cs:>10.3f}')

print(f'\nMonotonicity (composite QoE):')
for img_name, r in results.items():
    comp = r['composite']
    viol = sum(1 for i in range(len(comp)-1) if comp[i]>comp[i+1]+0.01)
    print(f'  {img_name:<20}: {viol} violations / {len(K_EVAL)-1} transitions  '
          f'({"MONOTONE" if viol==0 else f"~monotone ({viol} small violations)"})')

print(f'\nSSIM vs LPIPS agreement:')
for img_name, ag in agreement.items():
    print(f'  {img_name:<20}: r={ag["corr"]:.4f}')

print(f'\nKey findings for TCLA patent:')
ki32  = np.argmin(np.abs(k_arr-32));  c32  = np.mean([r['composite'][ki32]  for r in results.values()])
ki64  = np.argmin(np.abs(k_arr-64));  c64  = np.mean([r['composite'][ki64]  for r in results.values()])
ki128 = np.argmin(np.abs(k_arr-128)); c128 = np.mean([r['composite'][ki128] for r in results.values()])
ki256 = np.argmin(np.abs(k_arr-256)); c256 = np.mean([r['composite'][ki256] for r in results.values()])
print(f'  Base layer QoE (k=32):   {c32:.3f}  — above freeze threshold')
print(f'  Mid-range QoE  (k=64):   {c64:.3f}')
print(f'  High-quality   (k=128):  {c128:.3f}')
print(f'  Full quality   (k=256):  {c256:.3f}')
print(f'  Total dynamic range:     {c256-c32:.3f}  — ({(c256-c32)/c256*100:.0f}% of full quality)')

from google.colab import files
files.download('/content/odp_analysis.png')
files.download('/content/odp_reconstruction_strip.png')
print('\nDownloaded both figures.')